## E1 - Phân loại Softmax, MLP, CNN

- Dataset lựa chọn: **Fashion MNIST**
- Đặc điểm dataset Fashion-MNIST:
  - Gồm 70,000 ảnh grayscale
  - Kích thước: $28 \times 28$
  - Số ảnh train: 60,000
  - Số ảnh test: 10,000
  - Số lớp phân loại: 10

|Label|Name|
|-|-|
|0|T-shirt/top|
|1|Trouser|
|2|Pullover|
|3|Dress|
|4|Coat|
|5|Sandal|
|6|Shirt|
|7|Sneaker|
|8|Bag|
|9|Ankle boot|

In [ ]:
import os
import random
import struct
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from typing import Tuple
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# Common function

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Config

CONFIG = {
    "dataset": "FashionMNIST",
    "data_dir": Path("../fashion-mnist"),
    "seed": 42,
    "val_ratio": 0.1, # 10%
    "batch_size": 128,
    "epochs": 20,
    "optimizer": "adam",
    "lr": 1e-3,
    "weight_decay": 0.0,
    "metric": "val_accuracy"
}

set_seed(CONFIG["seed"])
DEVICE = get_device()

In [ ]:
# Fashion-MNIST Dataset supporter

def read_idx(filename: str) -> np.ndarray:
    """Read an IDX file and return it as a NumPy array."""
    with open(filename, "rb") as f:
        zero, data_type, dims = struct.unpack(">HBB", f.read(4))
        shape = tuple(struct.unpack(">I", f.read(4))[0] for _ in range(dims))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)


def load_fashion_mnist(data_dir: Path) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Load the Fashion-MNIST dataset from the specified directory following Github repository structure."""
    train_images = read_idx(data_dir / "train-images-idx3-ubyte")
    train_labels = read_idx(data_dir / "train-labels-idx1-ubyte")
    test_images = read_idx(data_dir / "t10k-images-idx3-ubyte")
    test_labels = read_idx(data_dir / "t10k-labels-idx1-ubyte")
    return train_images, train_labels, test_images, test_labels

train_images, train_labels, test_images, test_labels = load_fashion_mnist(CONFIG["data_dir"])

## The first speech.
## print the shapes of the loaded datasets
print("Train images shape:", train_images.shape)
print("Train labels shape:", train_labels.shape)
print("Test images shape:", test_images.shape)
print("Test labels shape:", test_labels.shape)

lb_set = np.unique(train_labels).tolist()
print("Unique labels in the training set:", lb_set)

print("Pixel range:", train_images.min(), "to", train_images.max())
print("Test pixel range:", test_images.min(), "to", test_images.max())

In [ ]:
# Fashion-MNIST configurations

## List of class names for the Fashion MNIST dataset
## according to Github
CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SHAPE = (1, 28, 28)             # (C, H, W)
INPUT_DIM = int(np.prod(IMAGE_SHAPE)) # flattened size

# print
print("Number of classes:", NUM_CLASSES)
print("Image shape:", IMAGE_SHAPE)
print("Input dimension (flattened):", INPUT_DIM)

In [ ]:
# Dataset split

SPLIT_PATH = Path(f"split_indices_seed{CONFIG['seed']}.npz")

if SPLIT_PATH.exists():
    # Reuse the saved split so later experiments use exactly the same samples
    with np.load(SPLIT_PATH) as split_indices:
        assert int(split_indices['seed']) == CONFIG['seed'], "Saved split seed does not match the current CONFIG['seed']"
        train_indices = split_indices['train_indices']
        val_indices = split_indices['val_indices']
else:
    # Assuming `dataset_size` is defined elsewhere
    all_indices = np.arange(len(train_labels))
    train_indices, val_indices = train_test_split(
        all_indices, 
        test_size=0.2, 
        random_state=CONFIG['seed'],
        stratify=train_labels  # Ensure the split maintains the label distribution
        shuffle=True,
    )
    train_indices, val_indices = np.sort(train_indices), np.sort(val_indices)
    np.savez(SPLIT_PATH, seed=CONFIG['seed'], train_indices=train_indices, val_indices=val_indices)

X_train, y_train = train_images[train_indices], train_labels[train_indices]
X_val, y_val = train_images[val_indices], train_labels[val_indices]
X_test, y_test = test_images, test_labels

## print shapes
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Plot support functions

## Location
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

## Define color palette for plots
BLUE, BLUE_DARK, GRAY = "#2a78d6", "#104281", "#c9c8c3"

## Update plt params
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#8a8984",
    "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "axes.grid": False,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8,
    "font.size": 9,
})

def save_fig(fig, name: str) -> None:
    fig.savefig(FIG_DIR / f"{name}.png")

In [ ]:
# Data exploration

SAMPLES_PER_CLASS = 8
rng = np.random.default_rng(CONFIG["seed"])

fig, axes = plt.subplots(NUM_CLASSES, SAMPLES_PER_CLASS, figsize=(SAMPLES_PER_CLASS * 0.9, NUM_CLASSES * 0.95))
for k, name in enumerate(CLASS_NAMES):
    picks = rng.choice(np.flatnonzero(y_train == k), SAMPLES_PER_CLASS, replace=False)
    for j, i in enumerate(picks):
        ax = axes[k, j]
        ax.imshow(X_train[i], cmap="gray", vmin=0, vmax=255)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[k, 0].set_ylabel(f"{k}: {name}", rotation=0, ha="right", va="center", fontsize=9)
fig.suptitle("Fashion-MNIST training samples (8 per class)", y=0.995)
fig.tight_layout()
save_fig(fig, "E0_samples_per_class")
plt.show()

In [ ]:
# Pixel statistics

X_train_f = X_train.astype(np.float32) / 255.0   # scale to [0, 1], as ToTensor() will
pixel_mean = X_train_f.mean()
pixel_std = X_train_f.std()
zero_frac = (X_train == 0).mean()

print(f"Pixel mean (train, [0,1] scale): {pixel_mean:.4f}")
print(f"Pixel std  (train, [0,1] scale): {pixel_std:.4f}")
print(f"Fraction of exactly-zero pixels : {zero_frac:.1%}")

NORM_MEAN, NORM_STD = float(pixel_mean), float(pixel_std)

In [ ]:
# Normalized pixel statistics

def preprocess(images: np.ndarray) -> torch.Tensor:
    # uint8 [N, 28, 28] -> normalized float32 [N, 1, 28, 28]
    x = torch.tensor(images, dtype=torch.float32).div(255.0).unsqueeze(1)
    return (x - NORM_MEAN) / NORM_STD

train_ds = TensorDataset(preprocess(X_train), torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(preprocess(X_val), torch.tensor(y_val, dtype=torch.long))
test_ds = TensorDataset(preprocess(X_test), torch.tensor(y_test, dtype=torch.long))

In [ ]:
# Dataloaders

def make_train_loader(seed: int) -> DataLoader:
    """A fresh generator with the same seed gives every model the same mini-batch order"""
    return DataLoader(
        train_ds, 
        batch_size=CONFIG["batch_size"], 
        shuffle=True,
        generator=torch.Generator().manual_seed(seed)
    )

# Unshuffled loaders for evaluation; train_eval_loader measures train accuracy in eval mode
train_eval_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"])
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"])
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"])

images, labels = next(iter(make_train_loader(CONFIG["seed"])))
flattened_images = images.flatten(start_dim=1)

print(f"Input data (B, C, H, W)   : ", tuple(images.shape))
print(f"Flattened input (B, C*H*W): ", tuple(flattened_images.shape))

In [ ]:
# Softmax Classifier

class SoftmaxClassifier(nn.Module):
    """Linear model on the flattened image: logits = W x + b."""
    
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)